# MaatProof Proof-of-Deploy Colab

This notebook runs the Python reference implementation for the formal certificate model `C = <P, E, pi, A>` and the acceptance rule `Accept(C) = WF(P) && Auth(E) && CheckR(pi, P, E) && Quorum(A)`.

In [ ]:
# In Google Colab, run this after cloning the repository:
# !git clone https://github.com/dngoins/MaatProof.git
# %cd MaatProof
# !pip install -e .

# In a local checkout, install with:
# !pip install -e .

In [ ]:
from pathlib import Path

from maatproof import (
    CertificateChecker,
    DeploymentCertificate,
    DeploymentPolicy,
    DeploymentRequest,
    EvidenceBundle,
    JsonlDeploymentLedger,
    PolicyPredicate,
    ProofDerivation,
    ProofStep,
    QuorumRule,
    ValidatorAttestation,
    signed_evidence,
    simulate_validators,
)

EVIDENCE_SECRET = b'evidence-secret'
VALIDATORS = {
    'validator-a': b'validator-a-secret',
    'validator-b': b'validator-b-secret',
    'validator-c': b'validator-c-secret',
}
NOW = 1_700_000_100.0

In [ ]:
def deployment_request(environment='production'):
    return DeploymentRequest(
        deployment_id='deploy-123',
        service='checkout',
        environment=environment,
        commit_sha='abc123',
        artifact_hash='sha256:artifact',
        requested_by='agent:planner',
    )


def deployment_policy(environment='production'):
    return DeploymentPolicy(
        policy_id='checkout-prod',
        version=1,
        environment=environment,
        freshness_seconds={'human_attestation': 3600, 'scan_report': 3600},
        predicates=[
            PolicyPredicate('test_passed', {'suite': 'unit'}),
            PolicyPredicate('vuln_count', {'severity': 'critical', 'operator': '<=', 'threshold': 0}),
            PolicyPredicate('environment_matches', {'target': environment}),
            PolicyPredicate('rollback_defined', {'service': 'checkout'}),
            PolicyPredicate('human_attested', {'role': 'release-manager'}),
        ],
    )


def evidence_bundle(request, include_scan=True, human_timestamp=NOW):
    objects = [
        signed_evidence('commit', 'commit_snapshot', {'deployment_id': request.deployment_id, 'commit_sha': request.commit_sha}, 'git', NOW, EVIDENCE_SECRET),
        signed_evidence('artifact', 'build_artifact', {'deployment_id': request.deployment_id, 'artifact_hash': request.artifact_hash}, 'builder', NOW, EVIDENCE_SECRET, dependencies=['commit']),
        signed_evidence('test', 'test_result', {'deployment_id': request.deployment_id, 'suite': 'unit', 'passed': True}, 'pytest', NOW, EVIDENCE_SECRET, dependencies=['artifact']),
        signed_evidence('env', 'environment_descriptor', {'deployment_id': request.deployment_id, 'environment': request.environment}, 'cluster', NOW, EVIDENCE_SECRET),
        signed_evidence('rollback', 'rollback_spec', {'deployment_id': request.deployment_id, 'service': request.service, 'rollback_plan': 'restore previous image'}, 'deploy-plan', NOW, EVIDENCE_SECRET),
        signed_evidence('human', 'human_attestation', {'deployment_id': request.deployment_id, 'role': 'release-manager', 'approved': True}, 'approvals', human_timestamp, EVIDENCE_SECRET),
    ]
    if include_scan:
        objects.append(signed_evidence('scan', 'scan_report', {'deployment_id': request.deployment_id, 'vulnerabilities': {'critical': 0, 'high': 1}}, 'scanner', NOW, EVIDENCE_SECRET, dependencies=['artifact']))
    return EvidenceBundle(objects)


def derivation(request, bad_rule=False):
    first_rule = 'TRUST_ME' if bad_rule else 'TEST_PASS'
    return ProofDerivation(
        final_conclusion=f'deploy_authorized:{request.deployment_id}',
        steps=[
            ProofStep('test-pass', first_rule, 'test_passed:unit', ['test']),
            ProofStep('scan-ok', 'VULN_OK', 'vuln_count:critical<=0', ['scan']),
            ProofStep('env-ok', 'ENVIRONMENT_MATCH', 'environment_matches', ['env']),
            ProofStep('rollback-ok', 'ROLLBACK_READY', 'rollback_defined', ['rollback']),
            ProofStep('human-ok', 'HUMAN_ATTESTED', 'human_attested:release-manager', ['human']),
            ProofStep('policy', 'POLICY_SATISFIED', 'policy_satisfied', premises=['test-pass', 'scan-ok', 'env-ok', 'rollback-ok', 'human-ok']),
            ProofStep('deploy', 'DEPLOY_AUTH', f'deploy_authorized:{request.deployment_id}', premises=['policy']),
        ],
    )


def certificate(include_scan=True, environment='production', policy_environment='production', human_timestamp=NOW, bad_rule=False):
    request = deployment_request(environment)
    return DeploymentCertificate(
        request=request,
        policy=deployment_policy(policy_environment),
        evidence=evidence_bundle(request, include_scan=include_scan, human_timestamp=human_timestamp),
        proof=derivation(request, bad_rule=bad_rule),
    )


def summarize(report):
    return {
        'accepted': report.accepted,
        'WF(P)': report.wf_policy,
        'Auth(E)': report.auth_evidence,
        'CheckR': report.check_proof,
        'Quorum(A)': report.quorum,
        'failures': [failure.code for failure in report.failures],
    }

## Happy path: valid certificate and finalized deployment

In [ ]:
valid = certificate()
pre_quorum_checker = CertificateChecker(EVIDENCE_SECRET, now=NOW)
valid.attestations = simulate_validators(valid, pre_quorum_checker, VALIDATORS, timestamp=NOW)

checker = CertificateChecker(EVIDENCE_SECRET, VALIDATORS, QuorumRule.majority(), now=NOW)
report = checker.accept(valid)
summarize(report)

In [ ]:
ledger = JsonlDeploymentLedger(Path('/tmp/maatproof-deployments.jsonl'))
entry = ledger.append(valid, report, timestamp=NOW)
entry.to_dict()

## Failure scenarios

In [ ]:
cases = {}

missing_scan = certificate(include_scan=False)
cases['missing scan evidence'] = checker.accept(missing_scan)

stale_human = certificate(human_timestamp=NOW - 7200)
cases['stale human attestation'] = checker.accept(stale_human)

wrong_environment = certificate(environment='staging', policy_environment='production')
cases['wrong environment binding'] = checker.accept(wrong_environment)

invalid_derivation = certificate(bad_rule=True)
cases['invalid derivation step'] = checker.accept(invalid_derivation)

insufficient_quorum = certificate()
digest = insufficient_quorum.digest()
insufficient_quorum.attestations = [
    ValidatorAttestation('validator-a', 'accept', digest, NOW).sign(VALIDATORS['validator-a']),
    ValidatorAttestation('validator-b', 'reject', digest, NOW).sign(VALIDATORS['validator-b']),
]
cases['insufficient quorum'] = checker.accept(insufficient_quorum)

{name: summarize(result) for name, result in cases.items()}